# 2026-06-08 FA 요인분석 - 매칭비발생

같은 기상셀·월·시간대의 매칭 비발생 대조군을 만들고, 산불 시점이 평상시보다 얼마나 위험 방향으로 이탈했는지를 FA 입력으로 준비한다.


# 데이터 준비

`jsw/강원_EDA/README.md`의 결합 방식을 따르되, 날씨 변수는 매칭 비발생 평균 대비 `위험방향_delta`로 변환한다.


## 0. 환경 설정

In [ ]:
from pathlib import Path
import gc
import math
import time
import warnings

import geopandas as gpd
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from shapely import wkt
from shapely.strtree import STRtree

warnings.filterwarnings("ignore")

try:
    from IPython.display import Markdown, display
except ImportError:
    display = print

    def Markdown(text):
        return text


font_path = "C:/Windows/Fonts/malgun.ttf"
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()

sns.set_theme(
    style="whitegrid",
    font=font_name,
    rc={
        "font.family": font_name,
        "font.sans-serif": [font_name],
        "axes.unicode_minus": False
    }
)

plt.rcParams["font.family"] = font_name
plt.rcParams["font.sans-serif"] = [font_name]
plt.rcParams["axes.unicode_minus"] = False

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
print("현재 matplotlib font.family:", plt.rcParams["font.family"])
print("현재 matplotlib font.sans-serif:", plt.rcParams["font.sans-serif"][:3])

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents, Path("D:/farm-system-public-02")]:
    if (candidate / "data").exists() and (candidate / "jsw").exists():
        ROOT = candidate
        break

WEATHER_DIR = ROOT / "data" / "강원도_날씨데이터"
FIRE_DIR = ROOT / "data" / "강원도_데이터"
SPATIAL_DIR = FIRE_DIR / "강원도_공간데이터"
FIRE_SPATIAL_DIR = FIRE_DIR / "산불_공간데이터"

TYPE_ORDER = ["영동 해안형", "영서 내륙형", "고지·산간형"]
ADDR_ORDER = ["임야번지(산)", "일반번지"]
CAUSE_ORDER = ["입산활동 발화 후보", "기타 인간활동 발화 후보"]
RANDOM_STATE = 20260608

print("ROOT:", ROOT)
print("matplotlib font:", plt.rcParams["font.family"])


## 1. 공통 함수

In [ ]:
def read_gpkg(path, columns=None):
    """pyogrio가 가능하면 사용하고, 아니면 geopandas 기본 엔진으로 읽는다."""
    try:
        return gpd.read_file(path, engine="pyogrio", columns=columns)
    except Exception:
        return gpd.read_file(path, columns=columns)


def valid_summary(df, columns):
    rows = []
    for col in columns:
        exists = col in df.columns
        rows.append({
            "변수": col,
            "전체행": len(df),
            "유효표본수": int(df[col].notna().sum()) if exists else 0,
            "결측수": int(df[col].isna().sum()) if exists else len(df),
            "결측률_pct": round(float(df[col].isna().mean() * 100), 2) if exists else 100.0,
        })
    return pd.DataFrame(rows)


def build_spatial_index(geoms):
    geom_arr = np.asarray(pd.Series(geoms).dropna().values, dtype=object)
    return geom_arr, STRtree(geom_arr)


def nearest_distance_from_index(points_gdf, geom_arr, tree, label):
    if len(geom_arr) == 0:
        return [np.nan] * len(points_gdf)
    out = []
    for pt in np.asarray(points_gdf.geometry.values, dtype=object):
        nearest = tree.nearest(pt)
        geom = geom_arr[int(nearest)] if isinstance(nearest, (int, np.integer)) else nearest
        out.append(pt.distance(geom))
    print(f"- {label} 최단거리 완료")
    return out


def add_rolling_weather(hourly):
    """현재 발생시각은 제외하고 직전 24/48/72시간 rolling 기상값을 만든다."""
    hourly = hourly.sort_values(["기상셀ID", "일시"]).reset_index(drop=True)
    shifted = hourly.groupby("기상셀ID", sort=False)[["풍속_m_s", "습도_pct", "강수량_mm"]].shift(1)
    grouped = shifted.groupby(hourly["기상셀ID"], sort=False)
    for window in [24, 48, 72]:
        hourly[f"직전{window}h_평균풍속"] = (
            grouped["풍속_m_s"].rolling(window, min_periods=1).mean().reset_index(level=0, drop=True)
        )
        hourly[f"직전{window}h_최소습도"] = (
            grouped["습도_pct"].rolling(window, min_periods=1).min().reset_index(level=0, drop=True)
        )
        hourly[f"직전{window}h_강수량합"] = (
            grouped["강수량_mm"].rolling(window, min_periods=1).sum().reset_index(level=0, drop=True)
        )
    return hourly


## 2. 산불 기본정보, 기상셀, 기후지형유형, 지형 결합

In [ ]:
t0 = time.time()

fire = pd.read_csv(FIRE_DIR / "강원도_산불발생.csv", encoding="utf-8-sig")
grid = gpd.read_file(WEATHER_DIR / "강원도날씨_격자.geojson").to_crs("EPSG:4326")
climate_type = pd.read_csv(WEATHER_DIR / "강원도날씨_기후지형유형_셀분류.csv", encoding="utf-8-sig")
topo = pd.read_csv(FIRE_SPATIAL_DIR / "강원도_산불_지형특성계산.csv", encoding="utf-8-sig")

fire["발생일시"] = pd.to_datetime(
    dict(year=fire["연도"], month=fire["월"], day=fire["일"]),
    errors="coerce",
) + pd.to_timedelta(fire["시간"].fillna(0).astype(int), unit="h")
fire["날짜"] = fire["발생일시"].dt.floor("D")
fire["월_key"] = fire["발생일시"].dt.month
fire["시간_key"] = fire["발생일시"].dt.hour
fire["산불위험계절"] = np.where(
    fire["월_key"].isin([11, 12, 1, 2, 3, 4, 5]),
    "산불위험계절(11~5월)",
    "기타계절",
)
fire["addr_type"] = np.where(
    fire["발생지역번지"].fillna("").astype(str).str.strip().str.startswith("산"),
    "임야번지(산)",
    "일반번지",
)
fire["발생시간대구분"] = pd.cut(
    fire["시간_key"],
    bins=[-0.1, 5, 9, 13, 17, 23],
    labels=["야간(0~5시)", "아침(6~9시)", "한낮(10~13시)", "오후(14~17시)", "저녁(18~23시)"],
)

fire_gdf = gpd.GeoDataFrame(
    fire,
    geometry=gpd.points_from_xy(fire["경도"], fire["위도"]),
    crs="EPSG:4326",
)
fire_cells = gpd.sjoin(
    fire_gdf,
    grid[["기상셀ID", "기후권역", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"])

unmatched_ids = fire_cells.loc[fire_cells["기상셀ID"].isna(), "fire_id"].tolist()
fire_cells = fire_cells[fire_cells["기상셀ID"].notna()].drop(columns=["geometry"]).copy()
fire_cells = fire_cells.merge(climate_type[["기상셀ID", "기후지형유형"]], on="기상셀ID", how="left")

fire_core = fire_cells.merge(topo.drop(columns=["위도", "경도"]), on="fire_id", how="left")
fire_core.loc[fire_core["TWI(지형다습지수)"] > 1e8, "TWI(지형다습지수)"] = np.nan
fire_core["사면방향_deg"] = (
    np.degrees(np.arctan2(fire_core["사면방향_sin"], fire_core["사면방향_cos"])) + 360
) % 360

for col in ["기후지형유형", "addr_type"]:
    if col == "기후지형유형":
        fire_core[col] = pd.Categorical(fire_core[col], categories=TYPE_ORDER, ordered=True)
    else:
        fire_core[col] = pd.Categorical(fire_core[col], categories=ADDR_ORDER, ordered=True)

print(f"산불 원자료: {len(fire):,}건")
print(f"기상셀 매칭 후: {len(fire_core):,}건")
print(f"미매칭 제외 fire_id: {unmatched_ids}")

del topo, fire_gdf, fire_cells
gc.collect()


## 3. 토지피복과 접근성 최단거리 결합

In [ ]:
land = read_gpkg(
    SPATIAL_DIR / "강원도_토지피복도_세분류_병합_1m.gpkg",
    columns=["L1_NAME", "L2_NAME"],
)
land["L1_NAME"] = land["L1_NAME"].fillna("미분류")
land["L2_NAME"] = land["L2_NAME"].fillna("미분류")

fire_points = gpd.GeoDataFrame(
    fire_core,
    geometry=gpd.points_from_xy(fire_core["경도"], fire_core["위도"]),
    crs="EPSG:4326",
).to_crs(land.crs)

fire_lc = gpd.sjoin(
    fire_points,
    land[["L1_NAME", "L2_NAME", "geometry"]],
    how="left",
    predicate="within",
).drop(columns=["index_right"])
fire_lc = fire_lc.drop_duplicates(subset="fire_id").reset_index(drop=True)
fire_lc["L1_NAME"] = fire_lc["L1_NAME"].fillna("미분류")
fire_lc["L2_NAME"] = fire_lc["L2_NAME"].fillna("미분류")

road = read_gpkg(SPATIAL_DIR / "강원도_병합_도로.gpkg", columns=[]).to_crs(land.crs)
trail_df = pd.read_csv(SPATIAL_DIR / "강원도_등산로.csv", encoding="utf-8-sig")
trail = gpd.GeoDataFrame(trail_df, geometry=trail_df["공간좌표"].apply(wkt.loads), crs="EPSG:4326").to_crs(land.crs)
imdo_df = pd.read_csv(SPATIAL_DIR / "강원도_임도망도.csv", encoding="utf-8-sig")
imdo = gpd.GeoDataFrame(imdo_df, geometry=imdo_df["공간좌표"].apply(wkt.loads), crs="EPSG:4326").to_crs(land.crs)

target_indices = {
    "도로": build_spatial_index(road.geometry),
    "임도": build_spatial_index(imdo.geometry),
    "등산로": build_spatial_index(trail.geometry),
    "산림지역": build_spatial_index(land.loc[land["L1_NAME"].eq("산림지역"), "geometry"]),
    "시가화건조지역": build_spatial_index(land.loc[land["L1_NAME"].eq("시가화건조지역"), "geometry"]),
    "농업지역": build_spatial_index(land.loc[land["L1_NAME"].eq("농업지역"), "geometry"]),
}

distance_specs = [
    ("도로", "도로_최단거리_m"),
    ("임도", "임도_최단거리_m"),
    ("등산로", "등산로_최단거리_m"),
    ("산림지역", "산림_최단거리_m"),
    ("시가화건조지역", "시가화_최단거리_m"),
    ("농업지역", "농업_최단거리_m"),
]
for label, col in distance_specs:
    geom_arr, tree = target_indices[label]
    fire_lc[col] = nearest_distance_from_index(fire_lc, geom_arr, tree, label)
    fire_lc[f"log1p_{col}"] = np.log1p(fire_lc[col])

fire_lc["생활권프록시"] = np.where(
    (fire_lc["도로_최단거리_m"] <= 30)
    | (fire_lc["시가화_최단거리_m"] <= 100)
    | (fire_lc["농업_최단거리_m"] <= 100),
    "도로/시가화/농업 인접",
    "비인접",
)
fire_lc["원인프록시_500m"] = np.where(
    (fire_lc["등산로_최단거리_m"] <= 500) | (fire_lc["임도_최단거리_m"] <= 500),
    "입산활동 발화 후보",
    "기타 인간활동 발화 후보",
)
fire_lc["원인프록시_500m"] = pd.Categorical(fire_lc["원인프록시_500m"], categories=CAUSE_ORDER, ordered=True)
fire_lc["기후지형_번지유형"] = (
    fire_lc["기후지형유형"].astype(str) + " / " + fire_lc["addr_type"].astype(str)
)

del fire_core, fire_points, road
gc.collect()

print("토지피복/접근성 결합 완료")


## 4. 발생시각·선행 날씨 결합

In [ ]:
daily = pd.read_csv(WEATHER_DIR / "강원도날씨_격자_일단위.csv", encoding="utf-8-sig")
hourly = pd.read_csv(WEATHER_DIR / "강원도날씨_격자_시간단위.csv", encoding="utf-8-sig")

daily["날짜"] = pd.to_datetime(daily["날짜"])
hourly["일시"] = pd.to_datetime(hourly["일시"])
hourly["날짜"] = hourly["일시"].dt.floor("D")
hourly["월_key"] = hourly["일시"].dt.month.astype("int8")
hourly["시간_key"] = hourly["일시"].dt.hour.astype("int8")
hourly = add_rolling_weather(hourly)

hourly_day = (
    hourly.groupby(["기상셀ID", "날짜"], as_index=False)
    .agg(
        최소습도_pct=("습도_pct", "min"),
        강수량합_mm=("강수량_mm", "sum"),
    )
)

hourly_features = hourly[
    [
        "기상셀ID", "일시", "날짜", "월_key", "시간_key",
        "풍속_m_s", "강수량_mm", "습도_pct",
        "직전24h_평균풍속", "직전24h_최소습도", "직전24h_강수량합",
        "직전48h_평균풍속", "직전48h_최소습도", "직전48h_강수량합",
        "직전72h_평균풍속", "직전72h_최소습도", "직전72h_강수량합",
    ]
].copy()
hourly_features = hourly_features.rename(
    columns={
        "풍속_m_s": "시점_풍속_m_s",
        "강수량_mm": "시점_강수량_mm",
        "습도_pct": "시점_습도_pct",
    }
)

daily_features = daily.rename(columns={"최대순간풍속_m_s": "당일_최대순간풍속_m_s"})
hourly_features = hourly_features.merge(
    daily_features[["기상셀ID", "날짜", "당일_최대순간풍속_m_s"]],
    on=["기상셀ID", "날짜"],
    how="left",
)

for lag in [1, 2, 3]:
    lag_df = hourly_day.copy()
    lag_df["날짜"] = lag_df["날짜"] + pd.Timedelta(days=lag)
    lag_df = lag_df.rename(
        columns={
            "최소습도_pct": f"D-{lag}_최소습도_pct",
            "강수량합_mm": f"D-{lag}_강수량합_mm",
        }
    )
    hourly_features = hourly_features.merge(lag_df, on=["기상셀ID", "날짜"], how="left")

float_cols = hourly_features.select_dtypes(include=["float64"]).columns
hourly_features[float_cols] = hourly_features[float_cols].astype("float32")

fire_lc = fire_lc.merge(
    hourly_features.rename(columns={"일시": "발생일시"}),
    on=["기상셀ID", "발생일시"],
    how="left",
    suffixes=("", "_weather"),
)

del daily, hourly, hourly_day, daily_features
gc.collect()

print(f"날씨 결합 완료: {time.time() - t0:.1f}초")
display(
    valid_summary(
        fire_lc,
        [
            "시점_습도_pct", "시점_풍속_m_s", "시점_강수량_mm",
            "직전24h_최소습도", "직전48h_최소습도", "직전72h_최소습도",
            "직전24h_평균풍속", "직전48h_평균풍속", "직전72h_평균풍속",
            "D-1_최소습도_pct", "D-2_최소습도_pct", "D-3_최소습도_pct",
            "고도(m)", "경사도(도)", "TPI(지형위치지수)", "TWI(지형다습지수)",
            "도로_최단거리_m", "임도_최단거리_m", "등산로_최단거리_m",
        ],
    )
)


## 5. 매칭 비발생 대조군과 FA 후보 데이터셋 구성

In [ ]:
META_COLS = [
    "fire_id", "발생일시", "기상셀ID", "기후권역", "기후지형유형",
    "addr_type", "기후지형_번지유형", "월_key", "시간_key", "산불위험계절",
    "발생시간대구분", "L1_NAME", "L2_NAME",
    "생활권프록시", "원인프록시_500m",
]

TOPO_FEATURE_COLS = [
    "고도(m)", "경사도(도)", "사면방향_sin", "사면방향_cos",
    "TPI(지형위치지수)", "TWI(지형다습지수)",
]

ACCESS_FEATURE_COLS = [
    "log1p_도로_최단거리_m", "log1p_임도_최단거리_m", "log1p_등산로_최단거리_m",
    "log1p_산림_최단거리_m", "log1p_시가화_최단거리_m", "log1p_농업_최단거리_m",
]

MATCHED_WEATHER_SPECS = [
    ("시점_습도_pct", "발생시각_습도_위험방향_delta", False),
    ("직전24h_최소습도", "직전24h_최소습도_위험방향_delta", False),
    ("직전48h_최소습도", "직전48h_최소습도_위험방향_delta", False),
    ("직전72h_최소습도", "직전72h_최소습도_위험방향_delta", False),
    ("D-1_최소습도_pct", "D-1_최소습도_위험방향_delta", False),
    ("D-2_최소습도_pct", "D-2_최소습도_위험방향_delta", False),
    ("D-3_최소습도_pct", "D-3_최소습도_위험방향_delta", False),
    ("시점_풍속_m_s", "발생시각_풍속_위험방향_delta", True),
    ("직전24h_평균풍속", "직전24h_평균풍속_위험방향_delta", True),
    ("직전48h_평균풍속", "직전48h_평균풍속_위험방향_delta", True),
    ("직전72h_평균풍속", "직전72h_평균풍속_위험방향_delta", True),
    ("당일_최대순간풍속_m_s", "발생일_최대순간풍속_위험방향_delta", True),
    ("시점_강수량_mm", "발생시각_강수량_위험방향_delta", False),
    ("직전24h_강수량합", "직전24h_강수량합_위험방향_delta", False),
    ("직전48h_강수량합", "직전48h_강수량합_위험방향_delta", False),
    ("직전72h_강수량합", "직전72h_강수량합_위험방향_delta", False),
    ("D-1_강수량합_mm", "D-1_강수량합_위험방향_delta", False),
    ("D-2_강수량합_mm", "D-2_강수량합_위험방향_delta", False),
    ("D-3_강수량합_mm", "D-3_강수량합_위험방향_delta", False),
]


def make_matched_controls(fire_df, hourly_feature_df, n_per_fire=5):
    fire_keys = pd.MultiIndex.from_frame(
        fire_df[["기상셀ID", "발생일시"]].rename(columns={"발생일시": "일시"})
    )
    candidates = hourly_feature_df.copy().reset_index(drop=True)
    hourly_keys = pd.MultiIndex.from_frame(candidates[["기상셀ID", "일시"]])
    candidates = candidates.loc[~hourly_keys.isin(fire_keys)].reset_index(drop=True)
    group_indices = candidates.groupby(["기상셀ID", "월_key", "시간_key"], sort=False).indices
    rng = np.random.default_rng(RANDOM_STATE)
    parts = []
    for _, row in fire_df.iterrows():
        key = (row["기상셀ID"], row["월_key"], row["시간_key"])
        idxs = group_indices.get(key)
        if idxs is None or len(idxs) == 0:
            continue
        take = rng.choice(idxs, size=min(n_per_fire, len(idxs)), replace=False)
        part = candidates.iloc[take].copy()
        part["fire_id"] = row["fire_id"]
        part["source_fire_time"] = row["발생일시"]
        part["control_rank"] = np.arange(1, len(part) + 1)
        parts.append(part)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


matched_controls = make_matched_controls(fire_lc, hourly_features, n_per_fire=5)

matched_keys = pd.MultiIndex.from_frame(matched_controls[["기상셀ID", "일시"]])
fire_keys_check = pd.MultiIndex.from_frame(
    fire_lc[["기상셀ID", "발생일시"]].rename(columns={"발생일시": "일시"})
)
matched_overlap = bool(matched_keys.isin(fire_keys_check).any())

available_meta_cols = [col for col in META_COLS if col in fire_lc.columns]
base_cols = available_meta_cols + [col for col in TOPO_FEATURE_COLS + ACCESS_FEATURE_COLS if col in fire_lc.columns]
fa_source = fire_lc[base_cols].copy()

matched_delta_feature_cols = []
for source_col, delta_col, higher_is_risk in MATCHED_WEATHER_SPECS:
    if source_col not in fire_lc.columns or source_col not in matched_controls.columns:
        continue
    control_mean = matched_controls.groupby("fire_id")[source_col].mean()
    fire_value = pd.to_numeric(fire_lc[source_col], errors="coerce")
    control_value = fire_lc["fire_id"].map(control_mean)
    if higher_is_risk:
        fa_source[delta_col] = fire_value - control_value
    else:
        fa_source[delta_col] = control_value - fire_value
    matched_delta_feature_cols.append(delta_col)

MATCHED_FA_FEATURE_COLS = (
    matched_delta_feature_cols
    + [col for col in TOPO_FEATURE_COLS if col in fa_source.columns]
    + [col for col in ACCESS_FEATURE_COLS if col in fa_source.columns]
)

for col in MATCHED_FA_FEATURE_COLS:
    fa_source[col] = pd.to_numeric(fa_source[col], errors="coerce")

fa_X_raw = fa_source[MATCHED_FA_FEATURE_COLS].copy()
fa_X_complete = fa_X_raw.dropna(axis=0, how="any").copy()
fa_meta_complete = fa_source.loc[fa_X_complete.index, available_meta_cols].copy()

summary = pd.DataFrame(
    {
        "항목": [
            "기상셀 매칭 산불",
            "매칭 비발생 행 수",
            "산불 1건당 평균 매칭 수",
            "산불 발생시점과 대조군 중복 여부",
            "FA 후보 변수 수",
            "FA 완전케이스 행 수",
        ],
        "값": [
            len(fire_lc),
            len(matched_controls),
            round(len(matched_controls) / len(fire_lc), 2),
            matched_overlap,
            len(MATCHED_FA_FEATURE_COLS),
            len(fa_X_complete),
        ],
    }
)

display(summary)
display(Markdown("### 매칭 비발생 기반 FA 후보 변수 결측 요약"))
display(valid_summary(fa_source, MATCHED_FA_FEATURE_COLS).sort_values("결측률_pct", ascending=False))

print("생성 객체:")
print("- fire_lc: EDA 방식 결합 완료 산불 테이블")
print("- matched_controls: 같은 기상셀·월·시간대 매칭 비발생 대조군")
print("- fa_source: 메타 컬럼 + 위험방향 delta/지형/접근성 FA 후보 변수")
print("- fa_X_raw: 매칭 비발생 기반 FA 후보 변수 원자료")
print("- fa_X_complete: 결측 없는 매칭 비발생 기반 FA 입력 후보")


# FA

여기서부터 `fa_X_complete`를 기준으로 매칭 비발생 기반 요인분석을 진행한다.
